# PaceAlgo ML — Notebook 08: FX+Gold Deep Validation + Pine-Ready Retrain

**Zweck:** Verifizieren dass der PF-1.34-Edge aus Exp 4 (FX+Gold-only) **robust** ist UND **Pine-budget-kompatibel** trainiert werden kann.

**4 Validierungen:**

1. **Per-Symbol Breakdown** — Trägt jedes der 3 Trainings-Symbole (EUR, USDJPY, XAU) zum Edge bei, oder nur eins?
2. **Per-Jahr Stability** — Hält PF über die Jahre (2020-2025) oder ist es ein Glück bestimmter Phasen?
3. **GBPUSD Hold-Out Test** — Generalisiert das Modell auf einen FX-Markt den es nie gesehen hat?
4. **Pine-Budget Retrain** — Modell mit max_depth=3, ≤30 trees, top-15 Features. Übersteht der Edge die Reduktion?

**Decision:**
- Alle 4 Checks bestanden → **V1 Pine Export bereit**
- Per-Symbol/Jahr instabil → Modell ist Glück, nicht Edge
- Pine-Reduktion zerstört Edge → Modell zu komplex für Pine → Backend-Pfad

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    MODELS_DIR = ARTIFACTS / 'models'
    REPORTS_DIR = ARTIFACTS / 'reports'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED.glob('*.parquet'))) < (len(list(DRIVE_BACKUP.glob('*.parquet'))) if DRIVE_BACKUP.exists() else 0):
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_MODELS, ARTIFACTS_REPORTS
    MODELS_DIR = ARTIFACTS_MODELS
    REPORTS_DIR = ARTIFACTS_REPORTS

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]

In [ ]:
!pip install -q lightgbm 2>&1 | tail -1

import json
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt

from core.config import (
    FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END,
    PINE_BUDGET,
)
from core.train import (
    stack_symbols, get_feature_columns,
    walk_forward_split, binary_label_for_long,
    train_lgbm, trading_metrics_from_predictions, sweep_threshold,
    load_features_and_labels,
)
from core.analysis import confidence_percentile_sweep

R_VALUE = 1.5
TRAINING_SYMBOLS = FX_SYMBOLS + METAL_SYMBOLS  # EUR, GBP, JPY, XAU
# GBPUSD is hold-out — stack_symbols filters it via drop_holdout
print(f'Training symbols (after hold-out drop): {[s for s in TRAINING_SYMBOLS if s not in DEV_HOLDOUT_SYMBOLS]}')
print(f'Hold-out: {DEV_HOLDOUT_SYMBOLS}')
print(f'Pine budget: {PINE_BUDGET}')

## 2. Train Pine-Budget-Compliant Model

Aktuell winning model: max_depth=4, num_leaves=15, 400 trees, 37 features.
Pine budget: max_depth=3, max ~30 trees, max 15 features.

Wir trainieren beide Varianten und vergleichen ob der Edge die Reduktion übersteht.

In [ ]:
# SHAP-Top-15 Features (from NB 06)
TOP_15_SHAP = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
]

# Build FX+Gold dataset (excludes GBPUSD hold-out)
df = stack_symbols(DATA_PROCESSED, symbols=TRAINING_SYMBOLS, tfs=PRIMARY_TIMEFRAMES,
                    R=R_VALUE, drop_holdout=DEV_HOLDOUT_SYMBOLS)
print(f'FX+Gold stacked: {df.shape}')
print(f'Symbols in df: {df["symbol"].unique()}')

df_clean = df.dropna(subset=TOP_15_SHAP + ['label'])
train_df, val_df, test_df = walk_forward_split(df_clean, TRAIN_END, VAL_END)
print(f'\ntrain: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}')

X_train = train_df[TOP_15_SHAP]; y_train = binary_label_for_long(train_df['label'])
X_val = val_df[TOP_15_SHAP]; y_val = binary_label_for_long(val_df['label'])
X_test = test_df[TOP_15_SHAP]; y_test = binary_label_for_long(test_df['label'])

In [ ]:
# Variant A: WINNING setup from Exp 4 (max_depth 4, 400 trees, 37 features)
PARAMS_RESEARCH = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 15, 'max_depth': 4, 'min_data_in_leaf': 100,
    'learning_rate': 0.03, 'num_iterations': 400, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
}

# Variant B: Pine-budget-compliant (max_depth 3, 30 trees, 15 features)
# IMPORTANT FIX: Previous runs got truncated to best_iteration (2-6 trees)
# due to LightGBM's default behavior. We now train DIRECTLY with lgb.train
# and NO early stopping callback — guaranteed full 30-tree model.
PARAMS_PINE = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7,
    'max_depth': 3,
    'min_data_in_leaf': 200,
    'learning_rate': 0.05,
    'num_iterations': 30,
    'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
}

print('Training Variant A (Research — exceeds Pine budget)...')
model_research = train_lgbm(X_train, y_train, X_val, y_val, params=PARAMS_RESEARCH, early_stopping_rounds=30)
print(f'  trees: {model_research.num_trees()}, best_iter: {model_research.best_iteration}')

print('\nTraining Variant B (Pine-Budget-Compliant — FORCED 30 trees)...')
# Direct lgb.train, NO early stopping, NO best_iteration truncation
train_data_pine = lgb.Dataset(X_train, label=y_train)
val_data_pine = lgb.Dataset(X_val, label=y_val, reference=train_data_pine)
model_pine = lgb.train(
    PARAMS_PINE,
    train_data_pine,
    num_boost_round=30,
    valid_sets=[val_data_pine],
    valid_names=['val'],
    callbacks=[lgb.log_evaluation(period=0)],  # silent, no early stopping
)
print(f'  trees: {model_pine.num_trees()} (forced full budget)')
print(f'  best_iteration: {model_pine.best_iteration} (informational only, not used for predictions)')


## 3. Check 4: Does Pine-Budget Edge Survive?

In [ ]:
def evaluate(model, X, df_subset, R, label='?'):
    proba = model.predict(X)
    # Fixed threshold 0.40 (tuned on VAL in earlier notebooks)
    m = trading_metrics_from_predictions(df_subset['label'], proba, 0.40, tp_R=R)
    # Expanded percentile sweep — gives full edge concentration picture for tier design
    sweep = confidence_percentile_sweep(df_subset['label'], proba, tp_R=R,
                                          percentiles=[1, 2, 3, 5, 10, 20])
    print(f'{label}: PF@0.40={m["profit_factor"]:.3f} (n={m["n_trades"]:,}, WR={m["win_rate"]:.3f})')
    for _, row in sweep.iterrows():
        print(f'   top {int(row["top_pct"]):2d}%: PF={row["profit_factor"]:.3f} '
              f'(n={int(row["n_selected"]):,}, WR={row["win_rate"]:.3f})')
    return m, sweep, proba

print('=== RESEARCH MODEL (depth 4, 400 trees, 15 features) — TEST set ===')
mr, sr, pr_research = evaluate(model_research, X_test, test_df, R_VALUE, 'research')

print('\n=== PINE MODEL (depth 3, 30 trees, 15 features) — TEST set ===')
mp, sp, pr_pine = evaluate(model_pine, X_test, test_df, R_VALUE, 'pine')

print('\n=== Pine constraint cost ===')
print(f'  Research model PF: {mr["profit_factor"]:.3f}')
print(f'  Pine model PF:     {mp["profit_factor"]:.3f}')
print(f'  Edge survival rate: {mp["profit_factor"] / mr["profit_factor"]:.2%}')
if mp['profit_factor'] >= 1.25:
    print('  -> Pine model retains usable edge.')
elif mp['profit_factor'] >= mr['profit_factor'] * 0.85:
    print('  -> Pine model retains ~85% of research edge — acceptable.')
else:
    print('  -> Pine reduction kills edge. Backend deployment needed.')

# Side-by-side Pine vs Research percentile comparison (key table for tier design)
print('\n=== Edge concentration: Pine vs Research (TEST set, top-N% PF) ===')
combined = pd.DataFrame({
    'top_pct':     sp['top_pct'].values,
    'pine_n':      sp['n_selected'].values,
    'pine_pf':     sp['profit_factor'].values,
    'pine_wr':     sp['win_rate'].values,
    'research_pf': sr['profit_factor'].values,
    'research_wr': sr['win_rate'].values,
})
display(combined.round(4))


## 4. Check 1: Per-Symbol Breakdown

Wir nutzen das **Pine-Model** für alle weiteren Checks (das ist was wir exportieren werden).

In [ ]:
print('Per-symbol performance (Pine model, TEST set, threshold 0.40):')
print()
per_symbol = []
for sym in test_df['symbol'].unique():
    mask = test_df['symbol'] == sym
    sub = test_df[mask]
    sub_proba = pr_pine[mask.values]
    m = trading_metrics_from_predictions(sub['label'], sub_proba, 0.40, tp_R=R_VALUE)
    sweep = confidence_percentile_sweep(sub['label'], sub_proba, tp_R=R_VALUE, percentiles=[1, 5])
    top1 = sweep[sweep['top_pct'] == 1].iloc[0]
    per_symbol.append({
        'symbol': sym,
        'n_test_bars': len(sub),
        'n_trades_thr': int(m['n_trades']),
        'pf_thr': float(m['profit_factor']),
        'wr_thr': float(m['win_rate']),
        'pf_top1': float(top1['profit_factor']),
        'wr_top1': float(top1['win_rate']),
    })
ps_df = pd.DataFrame(per_symbol)
display(ps_df.round(4))

## 5. Check 2: Per-Year Stability

In [ ]:
# Use ALL data (train + val + test) for per-year analysis to get full picture
all_proba = model_pine.predict(df_clean[TOP_15_SHAP])
all_df = df_clean.copy()
all_df['proba'] = all_proba
all_df['year'] = all_df.index.year

print('Per-year performance (Pine model, threshold 0.40):')
per_year = []
for year, sub in all_df.groupby('year'):
    m = trading_metrics_from_predictions(sub['label'], sub['proba'].values, 0.40, tp_R=R_VALUE)
    sweep = confidence_percentile_sweep(sub['label'], sub['proba'].values, tp_R=R_VALUE, percentiles=[1, 5])
    top1 = sweep[sweep['top_pct'] == 1].iloc[0] if not sweep.empty else None
    in_train = year < TRAIN_END.year
    per_year.append({
        'year': year,
        'in_train': in_train,
        'n_bars': len(sub),
        'n_trades': int(m['n_trades']),
        'pf_thr': float(m['profit_factor']),
        'wr_thr': float(m['win_rate']),
        'pf_top1': float(top1['profit_factor']) if top1 is not None else float('nan'),
        'wr_top1': float(top1['win_rate']) if top1 is not None else float('nan'),
    })
py_df = pd.DataFrame(per_year)
display(py_df.round(4))

# Highlight: are TEST years (>=2024) also positive?
print('\nTest-period years (out-of-sample):')
display(py_df[~py_df['in_train']].round(4))

## 6. Check 3: GBPUSD Hold-Out Test

In [ ]:
import pandas as _pd

# Robust tz conversion
val_end_ts = pd.Timestamp(VAL_END)
if val_end_ts.tz is None:
    val_end_ts = val_end_ts.tz_localize('UTC')
else:
    val_end_ts = val_end_ts.tz_convert('UTC')

print('GBPUSD hold-out — Pine model never saw this market during training')
print()
ho_rows = []
for tf in PRIMARY_TIMEFRAMES:
    ho_df = load_features_and_labels(DATA_PROCESSED, 'GBPUSD', tf, R_VALUE)
    if ho_df.empty:
        print(f'  SKIP GBPUSD {tf}: missing')
        continue
    ho_df = ho_df.dropna(subset=TOP_15_SHAP)
    ho_df = ho_df[ho_df.index >= val_end_ts]  # OOS window only
    if ho_df.empty:
        continue
    ho_proba = model_pine.predict(ho_df[TOP_15_SHAP])
    m = trading_metrics_from_predictions(ho_df['label'], ho_proba, 0.40, tp_R=R_VALUE)
    sweep = confidence_percentile_sweep(ho_df['label'], ho_proba, tp_R=R_VALUE, percentiles=[1, 5, 10])
    top1 = sweep[sweep['top_pct'] == 1].iloc[0]
    ho_rows.append({
        'symbol': 'GBPUSD', 'tf': tf,
        'n_bars': len(ho_df),
        'n_trades': int(m['n_trades']),
        'pf_thr': float(m['profit_factor']),
        'wr_thr': float(m['win_rate']),
        'pf_top1': float(top1['profit_factor']),
        'wr_top1': float(top1['win_rate']),
    })
    print(f'  GBPUSD {tf}: n_bars={len(ho_df):,}  PF@0.40={m["profit_factor"]:.3f}  '
           f'top1 PF={top1["profit_factor"]:.3f}')

ho_df_summary = pd.DataFrame(ho_rows)
print()
display(ho_df_summary.round(4))

if not ho_df_summary.empty:
    avg_pf = ho_df_summary['pf_thr'].mean()
    avg_top1 = ho_df_summary['pf_top1'].mean()
    print(f'\nGBPUSD Hold-out avg PF: {avg_pf:.3f}, avg Top-1% PF: {avg_top1:.3f}')
    if avg_pf >= 1.2:
        print('  -> GENERALIZES well to unseen FX market.')
    elif avg_pf >= 1.05:
        print('  -> Mild positive on unseen — accept with caveat.')
    else:
        print('  -> FAILS to generalize. Model is overfitted to EUR/JPY/XAU.')

In [ ]:
# Define proba cutoffs that map to confidence tiers based on percentile sweep
# (using Pine model proba distribution)
proba_sorted = np.sort(pr_pine)[::-1]  # descending
n_total = len(proba_sorted)

tier_definitions = [
    ('Standard',  10.0),   # top 10% of confidence
    ('High',      3.0),    # top 3%
    ('Premium',   1.0),    # top 1%
]

tier_rows = []
for name, pct in tier_definitions:
    n_at_tier = max(1, int(n_total * pct / 100))
    cutoff = float(proba_sorted[n_at_tier - 1])
    selected = pr_pine >= cutoff
    traded = test_df['label'].iloc[selected.nonzero()[0]]
    wins = int((traded == 1).sum())
    losses = int((traded == -1).sum())
    neutrals = int((traded == 0).sum())
    win_R = R_VALUE * 1.0
    pf = (wins * win_R) / losses if losses > 0 else float('inf') if wins > 0 else 0.0
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    tier_rows.append({
        'tier': name,
        'top_pct': pct,
        'proba_cutoff': cutoff,
        'n_trades_oos': wins + losses + neutrals,
        'wins': wins,
        'losses': losses,
        'win_rate': wr,
        'profit_factor': pf,
        'expected_R_per_trade': (wins * win_R - losses) / (wins + losses + neutrals) if (wins + losses + neutrals) > 0 else 0,
    })

tier_df = pd.DataFrame(tier_rows)
print('Tier preview (Pine model, TEST set):')
display(tier_df.round(4))

# How often does each tier trigger per day?
test_days = (test_df.index.max() - test_df.index.min()).days
print(f'\nTest period: {test_days} days')
for r in tier_rows:
    per_day = r['n_trades_oos'] / max(test_days, 1)
    print(f'  {r["tier"]:10s}: ~{per_day:.2f} signals/day across all FX+Gold symbols  (PF {r["profit_factor"]:.3f}, WR {r["win_rate"]*100:.1f}%)')


## 6.5 Confidence-Tier Design Preview (für späteres Marketing)

Sehen wir wie sich der Edge in **Confidence-Tiers** aufteilen lässt — für ein späteres User-Facing System mit Standard/High/Premium Signal-Labels, ohne dass interne Probabilities sichtbar werden.

**Tier-Cutoffs** sind hier datengetrieben aus dem Pine-Model auf TEST set abgeleitet.


## 7. Save Pine-Ready Model + Meta

In [ ]:
model_path = MODELS_DIR / 'lgbm_fxgold_pine.txt'
model_pine.save_model(str(model_path))
print(f'Pine model saved: {model_path} ({model_path.stat().st_size / 1024:.1f} KB)')

meta = {
    'model_name': 'PaceAlgo_FXGold_v1.0',
    'description': 'LightGBM for FX (EUR, JPY) + Gold (XAU), Pine-budget-compliant',
    'feature_cols': TOP_15_SHAP,
    'num_features': len(TOP_15_SHAP),
    'num_trees': model_pine.num_trees(),
    'max_depth': PARAMS_PINE['max_depth'],
    'best_iteration': int(model_pine.best_iteration) if model_pine.best_iteration else model_pine.num_trees(),
    'R': R_VALUE,
    'sl_atr_mult': 1.0,
    'time_barrier_bars': 24,
    'threshold': 0.40,
    'training_symbols': [s for s in TRAINING_SYMBOLS if s not in DEV_HOLDOUT_SYMBOLS],
    'holdout_symbols': DEV_HOLDOUT_SYMBOLS,
    'train_end': str(TRAIN_END),
    'val_end': str(VAL_END),
    'test_pf_threshold': float(mp['profit_factor']),
    'test_wr_threshold': float(mp['win_rate']),
    'test_n_trades': int(mp['n_trades']),
    'per_symbol_test': per_symbol,
    'gbpusd_holdout': ho_rows,
}
meta_path = MODELS_DIR / 'lgbm_fxgold_pine_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(f'Meta saved: {meta_path}')

# Also dump tree structure as JSON for Pine code generation later
tree_dump = model_pine.dump_model()
tree_path = MODELS_DIR / 'lgbm_fxgold_pine_trees.json'
with open(tree_path, 'w') as f:
    json.dump(tree_dump, f, indent=2, default=str)
print(f'Tree dump saved: {tree_path} ({tree_path.stat().st_size / 1024:.1f} KB)')
print(f'  -> ready for Pine code generation in NB 09')

## 8. Final Decision

In [ ]:
print('=' * 70)
print('FX+GOLD V1 — FINAL VALIDATION SUMMARY')
print('=' * 70)
print(f'Pine model: {model_pine.num_trees()} trees, depth {PARAMS_PINE["max_depth"]}, {len(TOP_15_SHAP)} features')
print(f'  -> Pine budget: max {PINE_BUDGET["max_trees"]} trees, depth {PINE_BUDGET["max_tree_depth"]}, {PINE_BUDGET["max_features_used"]} features')
print()
print(f'1) Pine reduction:    Research PF {mr["profit_factor"]:.3f} -> Pine PF {mp["profit_factor"]:.3f}')
print()
print(f'2) Per-symbol breakdown (TEST):')
for r in per_symbol:
    print(f'   {r["symbol"]}: PF {r["pf_thr"]:.3f} (n={r["n_trades_thr"]:,})')
print()
print(f'3) Per-year stability (full data):')
for r in per_year:
    flag = '(train)' if r['in_train'] else '(TEST)'
    print(f'   {r["year"]} {flag}: PF {r["pf_thr"]:.3f} (n={r["n_trades"]:,})')
print()
print(f'4) GBPUSD hold-out (never seen during training):')
for r in ho_rows:
    print(f'   GBPUSD {r["tf"]}: PF {r["pf_thr"]:.3f} (n={r["n_trades"]:,})')

print('\n' + '=' * 70)
if (mp['profit_factor'] >= 1.25 and
    all(r['pf_thr'] >= 1.1 for r in per_symbol) and
    sum(1 for r in per_year if r['pf_thr'] >= 1.0) >= len(per_year) * 0.7 and
    sum(1 for r in ho_rows if r['pf_thr'] >= 1.1) >= 1):
    print('VERDICT: ALL CHECKS PASSED -> proceed to Pine export (NB 09)')
else:
    print('VERDICT: Some checks weak. Review individually:')
    print('  Pine PF >= 1.25?    ', mp['profit_factor'] >= 1.25)
    print('  Each symbol PF>=1.1?', all(r['pf_thr'] >= 1.1 for r in per_symbol))
    print('  >=70% years OK?     ', sum(1 for r in per_year if r['pf_thr'] >= 1.0) >= len(per_year) * 0.7)
    print('  GBPUSD hold-out OK? ', sum(1 for r in ho_rows if r['pf_thr'] >= 1.1) >= 1)
print('=' * 70)